# ReLU、GELU、Sigmoid

**面试回答：**隐藏层 ReLU/GELU 关注梯度传播，输出 Sigmoid 提供二分类概率；不能混淆两者用途。

## 真实案例

客服风险模型对 8 个 logit 比较三种激活及其梯度。

In [1]:
import numpy as np  # 导入 NumPy 计算激活。
z=np.array([-5.,-2.,-1.,0.,1.,2.,5.,8.])  # 构造风险网络的输入 logit。
print('logit=',z.tolist())  # 输出输入样本。
print('样本数=',len(z))  # 输出样本规模。

logit= [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0, 8.0]
样本数= 8


## Baseline / 基线

Sigmoid 放在隐藏层会在大绝对值区间饱和。

In [2]:
sigmoid=1/(1+np.exp(-z))  # 计算 Sigmoid 输出。
sigmoid_grad=sigmoid*(1-sigmoid)  # 计算 Sigmoid 导数。
print('Sigmoid=',np.round(sigmoid,3))  # 输出基线激活。
print('Sigmoid梯度=',np.round(sigmoid_grad,4))  # 输出饱和梯度。

Sigmoid= [0.007 0.119 0.269 0.5   0.731 0.881 0.993 1.   ]
Sigmoid梯度= [0.0066 0.105  0.1966 0.25   0.1966 0.105  0.0066 0.0003]


In [3]:
relu=np.maximum(0,z)  # 计算 ReLU 输出。
relu_grad=(z>0).astype(float)  # 计算 ReLU 次梯度。
gelu=.5*z*(1+np.tanh(np.sqrt(2/np.pi)*(z+.044715*z**3)))  # 计算 GELU 近似输出。
print('ReLU=',np.round(relu,3))  # 输出 ReLU。
print('ReLU梯度=',relu_grad)  # 输出 ReLU 梯度。
print('GELU=',np.round(gelu,3))  # 输出 GELU。
print('输出层 Sigmoid 概率=',np.round(sigmoid,3))  # 强调 Sigmoid 的正确位置。

ReLU= [0. 0. 0. 0. 1. 2. 5. 8.]
ReLU梯度= [0. 0. 0. 0. 1. 1. 1. 1.]
GELU= [-0.    -0.045 -0.159  0.     0.841  1.955  5.     8.   ]
输出层 Sigmoid 概率= [0.007 0.119 0.269 0.5   0.731 0.881 0.993 1.   ]


## 结果解读

Sigmoid 的极端输入梯度接近零；ReLU 负区梯度为零；GELU 平滑门控。是否更好取决于结构、初始化和训练。

In [4]:
print('激活 | 负大输入 | 正大输入')  # 输出比较表头。
print('Sigmoid',round(float(sigmoid[0]),3),round(float(sigmoid[-1]),3))  # 输出 Sigmoid 饱和。
print('ReLU',relu[0],relu[-1])  # 输出 ReLU 结果。
print('GELU',round(float(gelu[0]),3),round(float(gelu[-1]),3))  # 输出 GELU 结果。
print('生产差距：需监控零激活比例、梯度范数、数值稳定和任务指标。')  # 说明边界。

激活 | 负大输入 | 正大输入
Sigmoid 0.007 1.0
ReLU 0.0 8.0
GELU -0.0 8.0
生产差距：需监控零激活比例、梯度范数、数值稳定和任务指标。


## 失败案例与修复

把负区 ReLU 单元永久置零会死亡；可尝试 Leaky ReLU 或检查初始化/学习率。

In [5]:
leaky=np.where(z>0,z,.01*z)  # 计算 Leaky ReLU 修复方案。
print('失败ReLU负区=',relu[:3])  # 输出死亡单元现象。
print('修复Leaky负区=',np.round(leaky[:3],3))  # 输出保留负区梯度的结果。
print('二分类头仍应使用 logit 加稳定交叉熵。')  # 说明输出层原则。

失败ReLU负区= [0. 0. 0.]
修复Leaky负区= [-0.05 -0.02 -0.01]
二分类头仍应使用 logit 加稳定交叉熵。


In [6]:
assert len(z)>=5  # 保护样本数。
assert sigmoid_grad[0]<.01  # 保护 Sigmoid 饱和现象。
assert relu[0]==0  # 保护 ReLU 负区归零。
assert leaky[0]<0  # 保护 Leaky 修复。